# House Price Prediction - Multiple Linear Regression (Gradient Descent)

In this notebook, we train our multiple linear regression model using **Stochastic Gradient Descent** on the 7 selected features.

### Why Scaling is Critical with Multiple Features:
In this model, our features have vastly different ranges:
- `sqft_lot`: spans from 520 to over 1,600,000 sqft
- `sqft_living`: spans from 290 to 13,540 sqft
- `grade`: spans from 1 to 13
- `waterfront`: is binary (0 or 1)

Without feature scaling, features with huge numbers would dominate the gradient updates, while smaller features like `waterfront` and `grade` would barely move. 
By putting `StandardScaler` in front of `SGDRegressor`, every feature is centered around 0 with a standard deviation of 1, allowing gradient descent to balance all features fairly and converge smoothly.

## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load the Processed Dataset

In [2]:
house_data = pd.read_csv("../data/processed/kc_house_multiple_lr.csv")

In [3]:
house_data.head(10)

,sqft_living,bedrooms,floors,grade,sqft_lot,yr_built,waterfront,price
0,1180,3,1.0,7,5650,1955,0,221900.0
1,2570,3,2.0,7,7242,1951,0,538000.0
2,770,2,1.0,6,10000,1933,0,180000.0
3,1960,4,1.0,7,5000,1965,0,604000.0
4,1680,3,1.0,8,8080,1987,0,510000.0
5,5420,4,1.0,11,101930,2001,0,1225000.0
6,1715,3,2.0,7,6819,1995,0,257500.0
7,1060,3,1.0,7,9711,1963,0,291850.0
8,1780,3,1.0,7,7470,1960,0,229500.0
9,1890,3,2.0,7,6560,2003,0,323000.0


In [4]:
house_data.isna().sum()

sqft_living    0
bedrooms       0
floors         0
grade          0
sqft_lot       0
yr_built       0
waterfront     0
price          0
dtype: int64

In [5]:
house_data.duplicated().sum()

np.int64(8)

## 3. Train-Test Split

Using the same 80/20 partition to keep evaluation fair and consistent.

In [6]:
X = house_data.iloc[:, :-1]
X

,sqft_living,bedrooms,floors,grade,sqft_lot,yr_built,waterfront
0,1180,3,1.0,7,5650,1955,0
1,2570,3,2.0,7,7242,1951,0
2,770,2,1.0,6,10000,1933,0
3,1960,4,1.0,7,5000,1965,0
4,1680,3,1.0,8,8080,1987,0
...,...,...,...,...,...,...,...
21608,1530,3,3.0,8,1131,2009,0
21609,2310,4,2.0,8,5813,2014,0
21610,1020,2,2.0,7,1350,2009,0
21611,1600,3,2.0,8,2388,2004,0


In [7]:
y = house_data["price"]
y

0        221900.0
1        538000.0
2        180000.0
3        604000.0
4        510000.0
           ...   
21608    360000.0
21609    400000.0
21610    402101.0
21611    400000.0
21612    325000.0
Name: price, Length: 21613, dtype: float64

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Build and Train the Pipeline

We combine `StandardScaler` and `SGDRegressor`:
- `StandardScaler()`: Standardizes all 7 features.
- `loss="squared_error"`: Standard least squares loss function.
- `penalty=None`: No L1/L2 penalties so we can compare directly with OLS.
- `learning_rate="invscaling"`: Decreases step size over iterations to prevent overshooting.
- `eta0=0.01`: Starting learning rate.

In [9]:
model = make_pipeline(
    StandardScaler(),
    SGDRegressor(
         loss="squared_error",
        penalty=None,
        learning_rate="invscaling",
        eta0=0.01,
        power_t=0.25,
        max_iter=1000,
        random_state=42
    )
)

In [10]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('sgdregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['sqft_living','bedrooms','floors',...,'sqft_lot','yr_built','waterfront']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


## 5. Model Evaluation on Test Data

In [11]:
y_pred = model.predict(X_test)
y_pred

array([ 513156.01507605,  652194.77687432, 1168548.17611298, ...,
        506167.05270115,  591623.97408551,  526637.02097985],
      shape=(4323,))

In [12]:
mae = mean_absolute_error(y_test, y_pred)
mae

144951.52313158067

In [13]:
mse = mean_squared_error(y_test, y_pred)
mse

54423408649.13688

In [14]:
rmse = np.sqrt(mse)
rmse

np.float64(233288.2522741702)

In [15]:
ordinary_r2_score = r2_score(y_test, y_pred)
ordinary_r2_score

0.6400013743077779

In [16]:
n = len(y_test)
p = X_test.shape[1]

adjusted_r2 = 1 - (
    (1 - ordinary_r2_score) * (n - 1)
    / (n - p - 1)
)

adjusted_r2

0.6394173672672575

In [17]:
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("Ordinary R²:", ordinary_r2_score)
print("Adjusted R²:", adjusted_r2)

MAE: 144951.52313158067
MSE: 54423408649.13688
RMSE: 233288.2522741702
Ordinary R²: 0.6400013743077779
Adjusted R²: 0.6394173672672575


### How do the results compare with Multiple OLS?
- **R2 of 0.6400**: Very close to OLS (0.6433).
- **MAE ($144,952)**: Actually slightly lower than OLS ($146,474), showing that SGD handled some of the extreme outliers very effectively.
- **RMSE ($233,288)**: Very close to OLS ($232,223).
This demonstrates that Gradient Descent scales gracefully to multiple features when properly standardized.

## 6. Relative Feature Importance from Scaled Weights

Because all 7 features were scaled to the exact same range (mean 0, std 1), we can compare the size of their coefficients directly to see which features impact price the most.

## 🔍 7. Standardized Coefficients & Relative Feature Importance

A major benefit of training with `StandardScaler` is that all input features share unit variance ($\sigma = 1$). Consequently, the magnitude of each coefficient $|w_j|$ directly reflects its **relative predictive importance**, free of unit scale discrepancies.

### What the feature weights tell us:
1. **`grade` (+149k) and `sqft_living` (+128k)** have the highest positive weights. Construction quality and living space are by far the two strongest drivers of house value.
2. **`waterfront` (+54k)** is strongly positive despite only a tiny fraction of homes having it.
3. **`yr_built` (-88k)** has a strong negative weight, meaning older homes tend to sell for less after accounting for quality and size.
4. **`bedrooms` (-37k)** is negative because holding total square footage fixed, cramming in extra bedrooms makes the house feel smaller.

## 7. Master Comparison Across All 4 Models

| Model | Method | Features Used | MAE | RMSE | R2 Score | Notes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **1. Simple LR (OLS)** | Analytical formula | `sqft_living` | $177,868 | $276,559 | 0.4941 | Baseline single-feature model |
| **2. Simple LR (GD)** | Gradient Descent | `sqft_living` (scaled) | $179,025 | $276,504 | 0.4943 | Matches OLS almost exactly |
| **3. Multiple LR (OLS)** | Analytical formula | 7 features | $146,474 | $232,223 | **0.6433** | +15% explained variance; full assumption checks |
| **4. Multiple LR (GD)** | Gradient Descent | 7 features (scaled) | **$144,952** | $233,288 | 0.6400 | Lowest MAE; scalable for large datasets |

---

## 8. Summary of Project Takeaways

1. **Adding features makes a big difference**: Moving from simple regression (living area only) to multiple regression (adding grade, age, bedrooms, waterfront) boosted R2 from **0.49 to 0.64** and cut average prediction errors by over **$30,000**.
2. **OLS and Gradient Descent find the same solution**: In both simple and multiple regression, gradient descent converges to virtually identical performance as OLS.
3. **Feature scaling is essential for Gradient Descent**: While OLS handles raw units without issue, Gradient Descent requires standardization to converge smoothly when features have different ranges.
4. **Real-world data has caveats**: Both models showed non-constant variance (heteroscedasticity) and skewed residuals due to luxury homes, highlighting natural opportunities for future work like log-transforming price or using regularized models.